# NB2 — Estimation (Abrigo Rev-2 migration)

**Purpose.** This notebook executes the Rev-2 mean-β primary OLS+HAC(4) regression and the 14-row sensitivity ladder; specification tests live in NB3 (`03_tests_and_sensitivity.ipynb`).

**Upstream input.** NB1 panel-fingerprint JSON at `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json` (the byte-exact upstream contract emitted by NB1 §1) and the 14 Phase 5a panel parquets at `contracts/.scratch/2026-04-25-task110-rev2-data/` (`panel_row_01_primary.parquet` … `panel_row_14_wc_cpi_weights_sens.parquet`) plus the published Rev-5.3.2 gate verdict at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json`.

**Downstream consumers.** NB3 consumes the per-row estimates JSON emitted by NB2 §1–§6 (point-coefficient + HAC(4) SE record per spec-row + sensitivity-row); the 14-row resolution table renders into the auto-rendered README via the Jinja2 template.

**Pre-committed gate verdict.** FAIL (one-sided T3b on β̂_X_d). Reproduced byte-exact from `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json`; never re-estimated in this migration. Anti-fishing invariants `N_MIN=75`, `POWER_MIN=0.80`, `MDES_SD=0.40` and `MDES_FORMULATION_HASH=4940360dcd2987…cefa` are immutable through Rev-5.3.x.

---

## §0 — Notebook header + panel-load sanity



### Why-markdown (4-part citation block)

**Reference.**

- Rev-2 spec, autonomous track A, at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` (655 lines; §3 functional form / OLS+HAC(4) Newey–West; §6 T3b 90% one-sided gate definition; §11.A convex-payoff insufficiency caveat — mean-β is necessary-but-insufficient for option-like product pricing).
- NB1 panel-fingerprint manifest at `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json` (the byte-exact upstream contract emitted by NB1 §1; consumed in NB2 §1+ to drive the 14-row resolution ladder).
- Phase 5b published gate verdict at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` (SHA-256 emitted by NB1 §0; `gate_verdict = "FAIL"`; `β̂_X_d = -2.7987e-8`; `HAC(4) SE = 1.4234e-8`; `t-stat = -1.9662`; `n = 76`; `window = [2024-09-27, 2026-03-13]`).
- Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md` (the disposition that reclassifies the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606` as Minteo-fintech, OUT of Mento-native scope; Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d, not the Mento-native COPm hedge-demand surface at `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`).
- Phase 5a Data Engineer outputs at `contracts/.scratch/2026-04-25-task110-rev2-data/` (`data_dictionary.md` is the canonical column-name authority; `manifest.md` is the row-summary; `_audit_summary.json` is the machine-readable per-row audit consumed by NB1 §0).
- Y₃ inequality-differential design at `contracts/docs/superpowers/specs/2026-04-24-y3-inequality-differential-design.md` (4-country panel CO/BR/KE/EU; equal-weight aggregation; pre-registered WC-CPI weights 60/25/15).
- X_d carbon-basket design at `contracts/docs/superpowers/specs/2026-04-24-carbon-basket-xd-design.md` (carbon-basket user-volume series; partition rule `trader != BancorArbitrage` per `project_carbon_user_arb_partition_rule`).
- Project memory: `project_mdes_formulation_pin` (Cohen f² formulation hash `4940360dcd2987…cefa`; free-tuning MDES_SD upward is anti-fishing-banned); `project_y3_inequality_differential_design`; `project_abrigo_inequality_hedge_thesis`; `project_abrigo_convex_instruments_inequality`; `project_abrigo_mento_native_only` (β-corrigendum: Rev-2's Minteo-fintech X_d close-out triggers the Minteo-fintech scope-mismatch dispatch to Task 11.P β-track on the Mento-native COPm surface).

**Why used.** NB2 is the estimation layer of the Rev-2 migration. NB1 produced the panel-fingerprint manifest that pinned the byte-exact upstream contract (14 panels, row counts, methodology tags, date windows, gate-verdict SHA-256). NB2 §0 establishes the panel-load contract that all downstream NB2 §1–§6 estimation cells will consume: byte-exact load of `panel_row_01_primary.parquet`, deterministic seed pin via `env.pin_seed`, schema validation against the Phase 5a `data_dictionary.md`. The Rev-2 published estimates (β̂_X_d = −2.7987e−8, HAC(4) SE = 1.4234e−8, t-stat = −1.9662, n = 76) are byte-exact-immutable per Rev-5.3.x anti-fishing invariants — NB2's job is to **reproduce** them, not re-estimate. Per `feedback_notebook_citation_block`, this 4-part block is non-negotiable for every estimation / sensitivity decision; the trio-checkpoint discipline (`feedback_notebook_trio_checkpoint`) HALTs after the §0 trio so sub-task 9 (NB2 §1 primary OLS+HAC(4)) dispatches separately.

**Relevance to results.** Byte-exact panel-load with seed pin is the necessary condition for byte-exact regression-coefficient reproduction in NB2 §1. The Rev-2 published primary-row estimate is computed on a panel of n = 76 Friday-anchored weeks over `[2024-09-27, 2026-03-13]` with column set `(week_start, y3_value, copm_diff, brl_diff, kes_diff, eur_diff, x_d, vix_avg, oil_return, us_cpi_surprise, banrep_rate_surprise, fed_funds_weekly, intervention_dummy)` — every dimension matching the Phase 5a `data_dictionary.md` byte-for-byte. Any drift surfaces here as a panel-construction or seeding bug rather than propagating silently into the §1 OLS coefficient. The `pin_seed(seed=42)` call is invariant across NB1 / NB2 / NB3; the `PYTHONHASHSEED` propagates to nbconvert child processes only (per the `env.py` docstring caveat), but bootstrap RNG state in NB3 is what actually consumes the seed — NB2 §0 / §1 are deterministic by closed-form OLS + HAC(4), not seed-dependent.

**Connection to product.** NB2 is the calibration-layer notebook for the Abrigo simulator: the 14-row mean-β resolution ladder produces the linear-hedge calibration inputs (point-β, HAC(4) SE, n, gate decision per row) that the simulator consumes for first-stage payoff matching. Per Rev-2 spec §11.A, mean-β is necessary-but-insufficient for the convex (option-like) instruments Abrigo sells against macroeconomic shocks viewed through the inequality lens (`project_abrigo_convex_instruments_inequality`); Rev-3 ζ-group (quantile / GARCH-X / lower-tail / option-implied vol) is where convex-payoff fitness gets tested, deferred to Task 11.O.ζ-α per `project_compact_survival_2026_04_26`. The Rev-2 migration closes scope-mismatch close-out under Rev-5.3.5: the Minteo-fintech X_d at `0xc92e8fc2…` is reclassified out of Mento-native scope, and the canonical Mento-native COPm at `0x8A567e2a…` is the actual hedge-demand surface — its own panel + estimation will use this same notebook scaffolding pattern under Task 11.P.spec-β / Task 11.P.exec-β (β-track).



In [1]:
"""NB2 §0 — panel-load sanity (primary-row byte-exact contract).

Loads `panel_row_01_primary.parquet` from the Phase 5a Data Engineer output
directory at `contracts/.scratch/2026-04-25-task110-rev2-data/`, pins the
deterministic seed via `env.pin_seed(seed=42)`, and asserts byte-exact match
against the Phase 5a `data_dictionary.md` schema (n = 76 rows, Friday-anchored
window `[2024-09-27, 2026-03-13]`, 13 canonical columns). The estimation cells
in NB2 §1+ consume this same panel; this trio establishes the load contract
that downstream cells inherit.

Functional-Python style: frozen dataclasses, free pure functions, full typing,
no inheritance.
"""
from __future__ import annotations

import importlib.util
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Final

import duckdb
import pandas as pd

# ---- Locate env.py and load by file path (notebooks/abrigo_y3_x_d/ is not on sys.path) ----

def _locate_abrigo_dir() -> Path:
    """Find the abrigo_y3_x_d/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "contracts" / "notebooks" / "abrigo_y3_x_d"
        if (candidate / "env.py").is_file():
            return candidate
        candidate2 = parent / "notebooks" / "abrigo_y3_x_d"
        if (candidate2 / "env.py").is_file():
            return candidate2
    raise RuntimeError(f"Could not locate abrigo_y3_x_d/env.py starting from cwd={cwd}")


_ABRIGO_DIR: Final[Path] = _locate_abrigo_dir()
_ENV_PATH: Final[Path] = _ABRIGO_DIR / "env.py"
_spec = importlib.util.spec_from_file_location("abrigo_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
sys.modules["abrigo_env"] = env
_spec.loader.exec_module(env)

# ---- Pre-committed Phase 5a primary-row schema (binding upstream contract) ----

PRIMARY_PARQUET_NAME: Final[str] = "panel_row_01_primary.parquet"
PRIMARY_METHODOLOGY: Final[str] = "y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable"
PRIMARY_WINDOW: Final[tuple[str, str]] = ("2024-09-27", "2026-03-13")
PRIMARY_N: Final[int] = 76

# Canonical column set per Phase 5a `data_dictionary.md` §1–§4 (order-insensitive).
EXPECTED_COLUMNS: Final[frozenset[str]] = frozenset({
    # Index
    "week_start",
    # Outcome (LHS) + per-country diagnostics
    "y3_value", "copm_diff", "brl_diff", "kes_diff", "eur_diff",
    # Variable of interest (X_d)
    "x_d",
    # Control set γ_1 … γ_6
    "vix_avg", "oil_return", "us_cpi_surprise",
    "banrep_rate_surprise", "fed_funds_weekly", "intervention_dummy",
})

# Deterministic seed for any RNG-dependent downstream cell (NB3 bootstrap).
# Mirror the value used by NB1 trio cells; closed-form OLS + HAC(4) in NB2 §1
# is not seed-dependent, but pinning here matches the
# `feedback_notebook_citation_block` invariant and is required by sub-plan §C-8.
SEED: Final[int] = 42


@dataclass(frozen=True, slots=True)
class PanelLoadContract:
    """Sanity-check fingerprint of the loaded primary panel.

    Emitted byte-exact alongside the panel DataFrame so downstream cells in
    NB2 §1+ can assert continuity.
    """
    parquet_path: Path
    n_rows: int
    dt_min: str
    dt_max: str
    columns: tuple[str, ...]
    source_methodology: str


# ---- Pure functions (no inheritance, no globals mutated) ----

def _phase5a_data_dir() -> Path:
    return env._CONTRACTS_DIR / ".scratch" / "2026-04-25-task110-rev2-data"


def _read_primary_panel(parquet_path: Path) -> pd.DataFrame:
    """Read the primary parquet via DuckDB (read-only, no row mutation)."""
    if not parquet_path.is_file():
        raise FileNotFoundError(f"Phase 5a primary parquet missing: {parquet_path}")
    conn = duckdb.connect()
    try:
        return conn.execute(f"SELECT * FROM read_parquet('{parquet_path}')").fetchdf()
    finally:
        conn.close()


def _query_methodology(parquet_path: Path) -> str:
    """Resolve the source-methodology literal for the primary panel from the
    Phase 5a `_audit_summary.json` (the panel parquet itself omits the
    methodology column per the data-dictionary §6 metadata-not-on-row rule).
    """
    import json
    audit_path = parquet_path.parent / "_audit_summary.json"
    with audit_path.open() as fh:
        audit = json.load(fh)
    return audit["row_01_primary"]["y3_methodology"]


def build_panel_contract(data_dir: Path) -> tuple[pd.DataFrame, PanelLoadContract]:
    """Load the primary panel and emit its byte-exact fingerprint."""
    parquet_path = data_dir / PRIMARY_PARQUET_NAME
    df = _read_primary_panel(parquet_path)
    methodology = _query_methodology(parquet_path)
    dt_min = df["week_start"].min().strftime("%Y-%m-%d")
    dt_max = df["week_start"].max().strftime("%Y-%m-%d")
    contract = PanelLoadContract(
        parquet_path=parquet_path,
        n_rows=int(len(df)),
        dt_min=dt_min,
        dt_max=dt_max,
        columns=tuple(df.columns),
        source_methodology=methodology,
    )
    return df, contract


# ---- Execute the panel-load + seed-pin sanity check ----

# Pin deterministic RNG state. PYTHONHASHSEED applies to child processes only
# (env.py docstring caveat); numpy + Python random are pinned in this process.
env.pin_seed(seed=SEED)

_DATA_DIR = _phase5a_data_dir()
panel_primary, contract = build_panel_contract(_DATA_DIR)

# Byte-exact assertions vs. Phase 5a data_dictionary.md (the load-bearing contract)
assert contract.n_rows == PRIMARY_N, (
    f"Primary panel n_rows drift: got {contract.n_rows}, expected {PRIMARY_N}"
)
assert (contract.dt_min, contract.dt_max) == PRIMARY_WINDOW, (
    f"Primary window drift: got ({contract.dt_min}, {contract.dt_max}), "
    f"expected {PRIMARY_WINDOW}"
)
_actual_cols = frozenset(contract.columns)
assert _actual_cols == EXPECTED_COLUMNS, (
    f"Primary panel column-set drift:\n"
    f"  missing = {EXPECTED_COLUMNS - _actual_cols}\n"
    f"  extra   = {_actual_cols - EXPECTED_COLUMNS}"
)
assert contract.source_methodology == PRIMARY_METHODOLOGY, (
    f"Primary methodology drift: got {contract.source_methodology!r}, "
    f"expected {PRIMARY_METHODOLOGY!r}"
)

# Friday-anchored invariant per data_dictionary.md §1.1
_weekday_set = set(panel_primary["week_start"].dt.weekday.unique())
assert _weekday_set == {4}, (
    f"Friday-anchor invariant violated: weekdays observed = {sorted(_weekday_set)} "
    f"(expected {{4}} = Friday)"
)

# Emit panel-load summary (sanity print; downstream JSON emission lives in NB2 §1+)
print(f"Phase 5a data dir : {_DATA_DIR}")
print(f"Primary parquet   : {contract.parquet_path.name}")
print(f"n_rows            : {contract.n_rows}")
print(f"window            : [{contract.dt_min}, {contract.dt_max}]")
print(f"columns ({len(contract.columns)})    : {list(contract.columns)}")
print(f"source_methodology: {contract.source_methodology}")
print(f"seed (env.pin_seed): {SEED}")
print()
print("Panel head (first 2 rows):")
print(panel_primary.head(2).to_string(index=False))


Phase 5a data dir : /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/2026-04-25-task110-rev2-data
Primary parquet   : panel_row_01_primary.parquet
n_rows            : 76
window            : [2024-09-27, 2026-03-13]
columns (13)    : ['week_start', 'y3_value', 'copm_diff', 'brl_diff', 'kes_diff', 'eur_diff', 'x_d', 'vix_avg', 'oil_return', 'us_cpi_surprise', 'banrep_rate_surprise', 'fed_funds_weekly', 'intervention_dummy']
source_methodology: y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable
seed (env.pin_seed): 42

Panel head (first 2 rows):
week_start  y3_value  copm_diff  brl_diff  kes_diff  eur_diff          x_d  vix_avg  oil_return  us_cpi_surprise  banrep_rate_surprise  fed_funds_weekly  intervention_dummy
2024-09-27  0.014237   0.003570  0.012624       NaN  0.026519  4604.160458   15.804   -0.056576              0.0                 0.000              4.83                   1
2024-10-04 -0.005903  -0.004091 -0.001508       

### Interpretation

**Panel loaded byte-exact against the Phase 5a contract.** `panel_row_01_primary.parquet` reads as n = 76 Friday-anchored rows over `[2024-09-27, 2026-03-13]` with the 13 canonical columns enumerated in `data_dictionary.md` (`week_start`, `y3_value`, `copm_diff`, `brl_diff`, `kes_diff`, `eur_diff`, `x_d`, `vix_avg`, `oil_return`, `us_cpi_surprise`, `banrep_rate_surprise`, `fed_funds_weekly`, `intervention_dummy`); `source_methodology` resolves from `_audit_summary.json` to `y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (Rev-5.3.2 default-flip post-Task 11.O Step-0). Every assertion in this cell encodes a load-bearing acceptance criterion: any future drift to row count, window, column set, methodology tag, or Friday-anchor invariant surfaces here rather than propagating silently into the §1 OLS coefficient.

**Seed pinned via `env.pin_seed(seed=42)`.** The `random` and `numpy.random` global RNG state is now deterministic in this process; `PYTHONHASHSEED=42` propagates to nbconvert child processes only (per the `env.py` docstring caveat — intended behavior). NB2 §1 is closed-form OLS + HAC(4) Newey–West and is therefore deterministic by construction; the seed pin matters for NB3 bootstrap re-confirmation cells, but is established here so every downstream cell in the notebook inherits the same RNG state without re-pinning.

**Scope framing under Rev-5.3.5 (Minteo-fintech scope-mismatch close-out).** This panel was constructed against the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`; per the Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md`, that address is reclassified as Minteo-fintech (out of Mento-native scope per `project_abrigo_mento_native_only`). The canonical Mento-native `StableTokenCOP` address is `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`; estimation against the Mento-native COPm hedge-demand surface dispatches separately under Task 11.P β-track (Task 11.P.spec-β / Task 11.P.exec-β). NB2 reproduces the Rev-2 published Minteo-fintech X_d estimates byte-exact — so Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d, not as a test of the Mento-native COPm hedge-demand surface.

**Forward pointer.** NB2 §1 (primary OLS+HAC(4) Newey–West regression on the spec-§4.1 equation `y3_value ~ x_d + vix_avg + oil_return + us_cpi_surprise + banrep_rate_surprise + fed_funds_weekly + intervention_dummy`) follows in NB-α sub-task 9, dispatched separately per the trio-checkpoint discipline. The published reproduction target (β̂_X_d = −2.7987e−8, HAC(4) SE = 1.4234e−8, t-stat = −1.9662, n = 76, gate verdict FAIL) is byte-exact-immutable.
